# Explore the performance of a trained model outside the training range

In [ ]:
import pandas as pd
from DerivModel import FeedForwardNetwork, train_model
from BasketDataGen import value_basket
from DerivPlots import scatter_plot
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.optim as optim
import torch.nn.functional as F
from tqdm.notebook import tqdm, trange
import numpy as np
import matplotlib.pyplot as plt
import QuantLib as ql

In [ ]:
model_file = 'modelDMC.pt'

In [ ]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

In [ ]:
input_size = 28
num_hidden_layers = 4
neurons_per_layer = 300
model = FeedForwardNetwork(input_size, num_hidden_layers, neurons_per_layer).to(device)

In [ ]:
model = torch.load(model_file, map_location=device, weights_only=False)

In [ ]:
model.eval()

In [ ]:
filename_root = 'test_basket_mt'

df1 = pd.concat(
    [pd.read_csv(f"{filename_root}{i}.csv") for i in range(20)],
    ignore_index=False
)

# Columns that need cleaning
cols_to_fix = ["maturity", "option_value", "error_estimate", "process_time", "samples"]

for col in cols_to_fix:
    if col in df1.columns:
        df1[col] = (
            df1[col]
            .astype(str)              # ensure string
            .str.strip("[]")         # remove brackets
            .astype(float)           # convert to numeric
        )
df1e6 = df1[df1['samples'] == 100000]
df1e6 = df1e6.drop('samples', axis=1)

In [ ]:
y = df1e6['option_value']
X = df1e6.drop(['option_value', 'process_time', 'error_estimate'], axis=1)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
X_scaled

In [ ]:
X.head()

In [ ]:
column_names = X.columns.tolist()
column_names

In [ ]:
maturity = 365
stock_price0 = 150.0
vols0 = 0.5
stock_price1 = 150.0
vols1 = 0.5
stock_price2 = 150.0
vols2 = 0.5
stock_price3 = 150.0
vols3 = 0.5
stock_price4 = 150.0
vols4 = 0.5
stock_price5 = 150.0
vols5 = 0.5
rho_0_1 = 0.4
rho_0_2 = 0.4
rho_0_3 = 0.4
rho_0_4 = 0.4
rho_0_5 = 0.4
rho_1_2 = 0.4
rho_1_3 = 0.4
rho_1_4 = 0.4
rho_1_5 = 0.4
rho_2_3 = 0.4
rho_2_4 = 0.4
rho_2_5 = 0.4
rho_3_4 = 0.4
rho_3_5 = 0.4
rho_4_5 = 0.4

In [ ]:
input_dict = dict()

In [ ]:
# Iterate through the variable names and add them to the dictionary
for variable_name in column_names:
    # Use globals() to access the value associated with the variable name
    input_dict[variable_name] = globals().get(variable_name)

## Stock price override values

In [ ]:
stock_price_override = np.linspace(0.0, 400.0, 17, endpoint=True)

In [ ]:
stock_price_override

In [ ]:
input_df = pd.DataFrame([input_dict] * 17)

In [ ]:
input_df['stock_price0'] = stock_price_override

In [ ]:
input_scaled = scaler.transform(input_df)

In [ ]:
NN_res = model(torch.tensor(input_scaled, dtype=torch.float32).to(device))

In [ ]:
NN_res_np = NN_res.detach().cpu().numpy()

In [ ]:
interest_rate = 0.0
strike = 100.0
divs = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
mc_samples_size = [100000]

In [ ]:
MC_res = list()

In [ ]:
for index, row in input_df.iterrows():
    maturity = int(row['maturity'])
    stocks = [row['stock_price0'], row['stock_price1'], row['stock_price2'], 
              row['stock_price3'], row['stock_price4'], row['stock_price5']]
    vols = [row['vols0'], row['vols1'], row['vols2'], 
            row['vols3'], row['vols4'], row['vols5']]
    corr_matrix = ql.Matrix(6,6,0.4)
    corr_matrix[0][0] = 1.0
    corr_matrix[1][1] = 1.0
    corr_matrix[2][2] = 1.0
    corr_matrix[3][3] = 1.0
    corr_matrix[4][4] = 1.0
    corr_matrix[5][5] = 1.0
    npv = value_basket(interest_rate, maturity, strike, stocks, vols, divs, corr_matrix, mc_samples_size) 
    MC_res.append(npv[0]['option_value'])

In [ ]:
plt_filename = 'stock_pricednnmc.png'

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(stock_price_override, NN_res_np, color='black', linestyle='-', label='DNN values')
plt.plot(stock_price_override, MC_res, color='black', linestyle='--', label='MC values')
plt.xlabel('Stock Price')
plt.ylabel('Option NPV')
plt.legend()
plt.tight_layout()
plt.savefig(plt_filename, dpi=300)
plt.show()

## Volatility Override

In [ ]:
vol_override = np.linspace(0.0, 3.0, 13, endpoint=True)
input_df = pd.DataFrame([input_dict] * 13)
input_df['vols0'] = vol_override

input_scaled = scaler.transform(input_df)
NN_res = model(torch.tensor(input_scaled, dtype=torch.float32).to(device))
NN_res_np = NN_res.detach().cpu().numpy()
interest_rate = 0.0
strike = 100.0
divs = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
mc_samples_size = [100000]

In [ ]:
MC_res = list()

for index, row in input_df.iterrows():
    maturity = int(row['maturity'])
    stocks = [row['stock_price0'], row['stock_price1'], row['stock_price2'], 
              row['stock_price3'], row['stock_price4'], row['stock_price5']]
    vols = [row['vols0'], row['vols1'], row['vols2'], 
            row['vols3'], row['vols4'], row['vols5']]
    corr_matrix = ql.Matrix(6,6,0.4)
    corr_matrix[0][0] = 1.0
    corr_matrix[1][1] = 1.0
    corr_matrix[2][2] = 1.0
    corr_matrix[3][3] = 1.0
    corr_matrix[4][4] = 1.0
    corr_matrix[5][5] = 1.0
    npv = value_basket(interest_rate, maturity, strike, stocks, vols, divs, corr_matrix, mc_samples_size) 
    MC_res.append(npv[0]['option_value'])

In [ ]:
plt_filename = 'vol_dnnmc.png'

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(vol_override, NN_res_np, color='black', linestyle='-', label='DNN values')
plt.plot(vol_override, MC_res, color='black', linestyle='--', label='MC values')
plt.xlabel('Volatility')
plt.ylabel('Option NPV')
plt.legend()
plt.tight_layout()
plt.savefig(plt_filename, dpi=300)
plt.show()

## Correlation override

In [ ]:
corr_override = np.linspace(-1.0, 1.0, 21, endpoint=True)
input_df = pd.DataFrame([input_dict] * 21)
input_df['rho_0_1'] = corr_override

input_scaled = scaler.transform(input_df)
NN_res = model(torch.tensor(input_scaled, dtype=torch.float32).to(device))
NN_res_np = NN_res.detach().cpu().numpy()
interest_rate = 0.0
strike = 100.0
divs = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
mc_samples_size = [100000]

In [ ]:
MC_res = list()

for index, row in input_df.iterrows():
    maturity = int(row['maturity'])
    stocks = [row['stock_price0'], row['stock_price1'], row['stock_price2'], 
              row['stock_price3'], row['stock_price4'], row['stock_price5']]
    vols = [row['vols0'], row['vols1'], row['vols2'], 
            row['vols3'], row['vols4'], row['vols5']]
    corr_matrix = ql.Matrix(6,6,0.4)
    corr_matrix[0][1] = corr_matrix[1][0] = row['rho_0_1']
    corr_matrix[0][0] = 1.0
    corr_matrix[1][1] = 1.0
    corr_matrix[2][2] = 1.0
    corr_matrix[3][3] = 1.0
    corr_matrix[4][4] = 1.0
    corr_matrix[5][5] = 1.0
    npv = value_basket(interest_rate, maturity, strike, stocks, vols, divs, corr_matrix, mc_samples_size) 
    MC_res.append(npv[0]['option_value'])

In [ ]:
plt_filename = 'correl_dnnmc.png'

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(corr_override, NN_res_np, color='black', linestyle='-', label='DNN values')
plt.plot(corr_override, MC_res, color='black', linestyle='--', label='MC values')
plt.xlabel('Correlation')
plt.ylabel('Option NPV')
plt.legend()
plt.tight_layout()
plt.savefig(plt_filename, dpi=300)
plt.show()